In [3]:
# Container setup for nanoGPT
# git clone https://github.com/karpathy/nanoGPT.git

# Step 1: Environment Setup
# %pip install torch transformers datasets tqdm accelerate --quiet

import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from tqdm import tqdm
import sys

import os
import torch
os.chdir('/app')
print(os.getcwd())  # should print /app

# ✅ Make sure nanoGPT is on the Python path
#sys.path.append("/app/nanoGPT")

#from model import GPT, GPTConfig  # from Karpathy's nanoGPT
from nanoGPT.model import GPT, GPTConfig  # from Karpathy's nanoGPT

# Device check
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ----- Teacher: OpenAI GPT-2 small -----
teacher_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(teacher_name)
teacher_model = GPT2LMHeadModel.from_pretrained(teacher_name).to(device)
teacher_model.eval()

# ----- Student: nanoGPT configurable GPT -----
student_config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=128,   # context window
    n_layer=4,        # smaller than GPT-2 (12)
    n_head=4,
    n_embd=640,      # smaller than GPT-2 (768)
)

student_model = GPT(student_config).to(device)

print("✅ Teacher and Student models are ready.")

/app
Using device: cuda
number of parameters: 51.86M
✅ Teacher and Student models are ready.


In [5]:
import math
import torch
from transformers import GPT2LMHeadModel

@torch.no_grad()
def evaluate_ppl(model, tokenizer, text, device, block_size=1024):
    """Compute perplexity for both Hugging Face and nanoGPT models."""
    tokens = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"].to(device)
    n = tokens.size(1)
    stride = block_size
    lls = []

    for i in range(0, n - block_size, stride):
        inp = tokens[:, i : i + block_size]
        tgt = tokens[:, i + 1 : i + block_size + 1]

        if isinstance(model, GPT2LMHeadModel):
            # Hugging Face GPT-2 (teacher)
            out = model(inp, labels=tgt)
            loss = out.loss
        else:
            # nanoGPT-style model
            logits, loss = model(inp, tgt)

        lls.append(loss.item() * block_size)

    if not lls:  # handle very short text (shorter than block size)
        inp = tokens[:, :-1]
        tgt = tokens[:, 1:]
        if isinstance(model, GPT2LMHeadModel):
            out = model(inp, labels=tgt)
            mean_loss = out.loss.item()
        else:
            _, mean_loss = model(inp, tgt)
            mean_loss = mean_loss.item()
    else:
        mean_loss = sum(lls) / ((n // stride) * block_size)

    return math.exp(mean_loss)


@torch.no_grad()
def generate_text(model, tokenizer, device, prompt, max_new_tokens=50, temperature=0.8):
    """Generate text from either a Hugging Face GPT-2 or a nanoGPT model."""
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    for _ in range(max_new_tokens):
        context = input_ids[:, -128:]  # crop context for nanoGPT

        if isinstance(model, GPT2LMHeadModel):
            outputs = model(context)
            logits = outputs.logits
        else:
            logits, _ = model(context, None)

        logits = logits[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat((input_ids, next_token), dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


def evaluate_and_generate(student_model, teacher_model, tokenizer, text, device):
    """
    Use a single string as both:
    - the text for perplexity evaluation
    - the prompt for generation
    """
    print("🔍 Evaluating perplexities on the given text...")
    student_ppl = evaluate_ppl(student_model, tokenizer, text, device, block_size=128)
    teacher_ppl = evaluate_ppl(teacher_model, tokenizer, text, device, block_size=1024)

    print(f"📊 Teacher perplexity: {teacher_ppl:.2f}")
    print(f"📉 Student perplexity: {student_ppl:.2f}")

    print("\n✨ Generating text continuations...")
    student_text = generate_text(student_model, tokenizer, device, text)
    teacher_text = generate_text(teacher_model, tokenizer, device, text)

    print("\n🧠 Student output:\n", student_text)
    print("\n🎓 Teacher output:\n", teacher_text)

    return {
        "teacher_ppl": teacher_ppl,
        "student_ppl": student_ppl,
        "student_text": student_text,
        "teacher_text": teacher_text,
    }


In [6]:
# Step 2: Dataset Preparation & Tokenization (fixed padding issue)
from datasets import load_dataset
import torch

# 1️⃣ Load dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# 2️⃣ Combine training text
train_text = "\n\n".join(dataset["train"]["text"])

# 3️⃣ Fix GPT-2 tokenizer padding issue (GPT-2 doesn’t define a padding token by default because it was trained only on contiguous text without padding.)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Set pad_token to eos_token: {tokenizer.pad_token}")

# 4 Tokenize all text at once (no truncation)
encodings = tokenizer(train_text, return_tensors="pt", padding=False, truncation=False)
input_ids = encodings["input_ids"].to(device)
num_tokens = input_ids.size(1)
print(f"Total tokens in training text: {num_tokens}")

# 5 Define get_batch 
def get_batch(batch_size=8, block_size=128):
    # sample a (block_size + 1)-long slice, then split into x ([:-1]) and y ([1:])
    num_tokens = input_ids.size(1)
    starts = torch.randint(0, num_tokens - (block_size + 1), (batch_size,))
    chunks = [input_ids[:, s:s+block_size+1].squeeze(0) for s in starts]
    batch = torch.stack(chunks)                          # [B, block_size+1]
    x = batch[:, :-1].contiguous()                       # [B, block_size]
    y = batch[:,  1:].contiguous()                       # [B, block_size]
    return x.to(device), y.to(device)


# Test
xb, yb = get_batch()
print(f"Batch shape: {xb.shape}")

Set pad_token to eos_token: <|endoftext|>


Token indices sequence length is longer than the specified maximum sequence length for this model (2428601 > 1024). Running this sequence through the model will result in indexing errors


Total tokens in training text: 2428601
Batch shape: torch.Size([8, 128])


In [7]:
for i in range(4):
    print(f"xb[{i}]", xb[i][:10])
    print(f"yb[{i}]", yb[i][:10])
    print()

xb[0] tensor([  257,  5182,  1497,   422,   262,  1913, 47346,   286,  2180,  2168],
       device='cuda:0')
yb[0] tensor([ 5182,  1497,   422,   262,  1913, 47346,   286,  2180,  2168,   837],
       device='cuda:0')

xb[1] tensor([ 3668,  6153,   663,  5197,   286,  4347,  2488,    12,    31, 35851],
       device='cuda:0')
yb[1] tensor([ 6153,   663,  5197,   286,  4347,  2488,    12,    31, 35851,  2488],
       device='cuda:0')

xb[2] tensor([ 9765,  3841,   764,   770, 10273,  7533,  2753,   355,   663, 33600],
       device='cuda:0')
yb[2] tensor([ 3841,   764,   770, 10273,  7533,  2753,   355,   663, 33600,   366],
       device='cuda:0')

xb[3] tensor([  262, 14884,  3615,  6265,  1497,   422,   262,  7179,  3615,   764],
       device='cuda:0')
yb[3] tensor([14884,  3615,  6265,  1497,   422,   262,  7179,  3615,   764,   383],
       device='cuda:0')



In [8]:
results = evaluate_and_generate(
    student_model, teacher_model, tokenizer,
    "Once upon a time", device
)

# perplexity for teacher makes no sense due to evaluation mismatch (short context, padding, or misaligned labels).

🔍 Evaluating perplexities on the given text...


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


📊 Teacher perplexity: 290.98
📉 Student perplexity: 88626.84

✨ Generating text continuations...

🧠 Student output:
 Once upon a time edited finding Prince toured commenters Member receives challenge Icon Adv Bravo associationsRI rushertif sweater 625 sluggish Jake imagination medications @@ PumpkinOut Annalad stripped commer compare observationalTXNut Always win Thief Pas prosecutionsrell TraderactStock Our lets agreementswraetime homosexuality FX radi Hand

🎓 Teacher output:
 Once upon a time, when you are so dejected from feeling like you are contained, that your energy seems like it can barely take your breath out of you, you begin to wonder if it's your fault.

Your eyes wander to the refrigerator on your way


In [ ]:
# Step 3: Define distillation loss and training step
import torch.nn.functional as F

# Temperature for softening distributions
temperature = 2.0
alpha = 0.5  # balance between distillation and standard CE loss

# Example optimizer
optimizer = torch.optim.AdamW(student_model.parameters(), lr=3e-4)

def distillation_step(xb, yb):
    student_model.train()
    optimizer.zero_grad()

    with torch.no_grad():                      # teacher predicts next token for xb
        t_logits = teacher_model(xb).logits    # [B, T, V]

    s_logits, _ = student_model(xb, yb)        # [B, T, V]

    # KL (softened)
    teacher_probs = F.softmax(t_logits / temperature, dim=-1)
    student_log_probs = F.log_softmax(s_logits / temperature, dim=-1)
    kl_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (temperature**2)

    # CE on true next tokens
    ce_loss = F.cross_entropy(
        s_logits.view(-1, s_logits.size(-1)),
        yb.view(-1),
        ignore_index=tokenizer.pad_token_id
    )

    loss = alpha * kl_loss + (1 - alpha) * ce_loss
    loss.backward()
    optimizer.step()
    return loss.item(), kl_loss.item(), ce_loss.item()

# ✅ Test one batch
xb, yb = get_batch()
loss, kl, ce = distillation_step(xb, yb)
print(f"Loss: {loss:.4f} (KL={kl:.4f}, CE={ce:.4f})")


NameError: name 'torch' is not defined

In [10]:
# Step 4: Distillation Training Loop
from tqdm import trange
import time
import os

num_steps = 3000        # try a small number first, like 500–1000 for testing
log_interval = 50
save_interval = 250
save_dir = "distilled_checkpoints"
os.makedirs(save_dir, exist_ok=True)

print("🚀 Starting distillation training...")

loss_history = []

for step in trange(num_steps, desc="Distilling"):
    xb, yb = get_batch()

    loss, kl, ce = distillation_step(xb, yb)
    loss_history.append(loss)

    # Logging
    if (step + 1) % log_interval == 0:
        avg_loss = sum(loss_history[-log_interval:]) / log_interval
        print(f"Step {step+1:>5d} | loss={avg_loss:.4f} | KL={kl:.4f} | CE={ce:.4f}")

    # Save checkpoint
    if (step + 1) % save_interval == 0:
        ckpt_path = os.path.join(save_dir, f"student_step{step+1}.pt")
        torch.save(student_model.state_dict(), ckpt_path)
        print(f"💾 Saved checkpoint to {ckpt_path}")

print("✅ Distillation complete.")


🚀 Starting distillation training...


Distilling:   2%|▏         | 50/3000 [00:04<03:26, 14.26it/s]

Step    50 | loss=308.3774 | KL=534.0594 | CE=8.4377


Distilling:   3%|▎         | 101/3000 [00:12<06:46,  7.12it/s]

Step   100 | loss=252.7218 | KL=476.5667 | CE=7.6978


Distilling:   5%|▌         | 150/3000 [00:19<06:01,  7.89it/s]

Step   150 | loss=230.4211 | KL=464.8392 | CE=7.4236


KeyboardInterrupt: 

In [17]:
results = evaluate_and_generate(
    student_model, teacher_model, tokenizer,
    "Once upon a time", device
)

# perplexity for teacher makes no sense due to evaluation mismatch (short context, padding, or misaligned labels).

🔍 Evaluating perplexities on the given text...
📊 Teacher perplexity: 290.98
📉 Student perplexity: 353.99

✨ Generating text continuations...

🧠 Student output:
 Once upon a time of the South Korean forces, The Sukama in the Philippines, in the U.S, is an extravagant or financial and 80% have been distributed on the China, and the U.S., 304, $293 ) and a very sufficient

🎓 Teacher output:
 Once upon a time, I missed, and long since departed, an adventure of winning the title of Greatest Magician in the world, and of being in the room with the greatest wizard ever: Harry Potter. I dined with him at the Goodyear's Bist


In [1]:
import os
print(os.getcwd())


/app/sandbox


In [3]:
import os
os.chdir('/app')
print(os.getcwd())  # should print /app
print(os.path.exists('persisted_models/suffix/BPI12_gpt2.pth'))

/app
True


In [4]:
import sys
print(sys.executable)


/opt/conda/bin/python
